<a href="https://colab.research.google.com/github/Ivan-Mayabi/Knowledge-Engineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Knowledge Engineering**
**: Smart Traffic Light Navigation Agent**

****

*Adapted from [Zainabu Muti](https://www.linkedin.com/in/zainabu-muti-3789591b3/)* :)

**Step 1: Identify the Task**

Notebook was copied from Github repo so as to reduce amount of typing required.

The task is to help a car/self driving truck decide what to do based on the town that it is in

**Goal:** Make the best decision on the route to follow so as to get to Kisumu from Nairobi.

**Step 2: Assemble Relevant Knowledge**

The system needs to know:
*   The town you are in.
*   The best town you can get to from said town in order to get to your Goal

Uses the same directed graph from the original PS-Agents Assignment.<br>
Uses the Dijkstra's Algorithm to find out what is the best path and creates the map based off of that. (A map API does the exact same thing)


In [47]:
import pprint
import heapq

class directed_graph:
    """
    - This class specifies all the functionality required to define a python graph
    """

    def __init__(self,directed=True):
        """
            - This is the constructor that specifies the graph is automatically directed for our usecase
            - It also creates the dictionary that we will base all town connections in
        """
        self.directed=directed
        self.adjacent = {}

    def add_vertex(self,vertex):
        """
        - This adds a particular town as a vertex on the graph
            - Towns are only added if they do not already exist in the graph
            - Towns are automatically assigned to be connected to an empty list of neighbours
        """

        if vertex not in self.adjacent.keys():
            self.adjacent[vertex] = []

    def add_edge(self,v1,v2,time,distance,directed=True):
        """
        - This is how we add edges or road connections between different towns in the map
            - Before a connection is added, we make sure the vertex/town exists
            - We then specify that the town in connected to a separate town.
            - This is done as appending a list of values to the list of already present neighbours
        """
        self.add_vertex(v1)
        self.add_vertex(v2)

        self.adjacent[v1].append((v2,time,distance))

        # This is never executed but just goes to show that you can traverse the graph in two ways - to a forward town or back to a previous one
        if not directed:
            self.adjacent[v2].append((v1,time,distance))

    def full_dict(self):
        """
        - Was a debugging utility that just printed the dictionary in a pretty manner (pretty print)
        """
        pprint.pprint(self.adjacent)

    def build(self):
        """
        - Creates the entire graph
        - This is the more detailed graph that forces our trailer to stay on trunk roads
        """

        self.add_edge("Nairobi","Githurai","26m","16.3km")
        self.add_edge("Githurai","Ruiru","17m","9.6km")
        self.add_edge("Ruiru", "Juja", "14m", "9.1km")
        self.add_edge("Juja", "Thika", "20m", "13.3km")
        self.add_edge("Thika", "Othaya", "114m", "86.8km")
        self.add_edge("Thika", "Sagana", "66m", "63.9km")
        self.add_edge("Thika", "Wang'uru", "70m", "63.1km")
        self.add_edge("Sagana", "Karatina", "25m", "25.8km")
        self.add_edge("Karatina", "Nyeri", "30m", "24.7km")
        self.add_edge("Othaya", "Nyeri", "23m", "16.9km")
        self.add_edge("Wang'uru", "Kutus", "70m", "63.1km")
        self.add_edge("Kutus", "Embu", "32m", "19.3km")
        self.add_edge("Embu", "Chuka", "59m", "43.8km")
        self.add_edge("Chuka", "Chogoria", "23m", "15.8km")
        self.add_edge("Chogoria", "Igoji", "16m", "10.3km")
        self.add_edge("Igoji", "Nkubu", "24m", "16.7km")
        self.add_edge("Nkubu", "Meru", "31m", "16.6km")
        self.add_edge("Meru", "Nanyuki", "81m", "76.2km")
        self.add_edge("Nyeri", "Muiga", "29m", "20.8km")
        self.add_edge("Muiga", "Ndaragwa", "56m", "59.2km")
        self.add_edge("Nanyuki", "Nyeri", "70m", "61.0km")
        self.add_edge("Ndaragwa", "Nyahururu", "30m", "26.2km")
        self.add_edge("Nyahururu", "Bahati", "46m", "42.3km")
        self.add_edge("Bahati", "Nakuru", "39m", "21.0km")
        self.add_edge("Nyahururu", "Dondori", "45m", "36.0km")
        self.add_edge("Dondori", "Nakuru", "60m", "42.8km")
        self.add_edge("Nairobi", "Mai Mahiu", "85m", "53.8km")
        self.add_edge("Mai Mahiu", "Karagita", "43m", "31.1km")
        self.add_edge("Mai Mahiu", "Ntulelei", "55m", "60.3km")
        self.add_edge("Karagita", "Naivasha", "15m", "7.2km")
        self.add_edge("Naivasha", "Gilgil", "39m", "30.4km")
        self.add_edge("Gilgil", "Nakuru", "61m", "42.0km")
        self.add_edge("Ntulelei", "Narok", "30m", "28.8km")
        self.add_edge("Narok", "Bomet", "82m", "81.0km")
        self.add_edge("Bomet", "Keroka", "65m", "56.9km")
        self.add_edge("Keroka", "Keumbu", "20m", "14.3km")
        self.add_edge("Keumbu", "Kisii", "18m", "12.2km")
        self.add_edge("Kisii", "Rongo", "36m", "26.0km")
        self.add_edge("Rongo", "Homa Bay", "38m", "31.0km")
        self.add_edge("Homa Bay", "Kendu Bay", "33m", "31.2km")
        self.add_edge("Kendu Bay", "Ahero", "53m", "53.3km")
        self.add_edge("Kisii", "Oyugis", "38m", "25.3km")
        self.add_edge("Oyugis", "Sondu", "40m", "38.0km")
        self.add_edge("Sondu", "Ahero", "18m", "29.5km")
        self.add_edge("Ahero", "Kisumu", "32m", "23.5km")
        self.add_edge("Muhoroni", "Ahero", "51m", "37.7km")
        self.add_edge("Kapsoit", "Muhoroni", "42m", "28.6km")
        self.add_edge("Kericho", "Kapsoit", "16m", "11.3km")
        self.add_edge("Londiani", "Kericho", "52m", "49.0km")
        self.add_edge("Molo", "Londiani", "31m", "22.1km")
        self.add_edge("Rongai", "Molo", "73m", "46.7km")
        self.add_edge("Nakuru", "Rongai", "58m", "33.2km")

    def dijkstras(self,start_node):
        """
        - This is a classic implementation of dijkstras algorigthm
        - It starts by specifying 2 new dictionaries
            - distances => tracks the shortest path to get to a particular town
            - previous_nodes => tracks the previous node that was used to get to that town
            - we set the start point of the journey to have a distance of 0
        - We also specify a list called pq, which specifies all the new towns that we will traverse
        - So long as there is a town that has not yet been traversed in pq
            - We will pop the top one from the list using the heapq library (Allows us to know that the top item on the stack has the lowest value always )
            - We will then check if the distance already known to get to that town is shorter
                    - If it is we will not change it but will move on
            - If not we will check all the listed towns that are connected to current town
                - We will get a distance to be the current distance to get to the town and add the distance specified by the adjacency list
                - Compare the distance we found and what is currently present
                - If what we found is less, we will update both the previous towns and the distances
                - We will then push that town into pq
            - We will repeat this until the pq is empty.
        - We then return both the dictionary of all the shortest distances as well as the towns that are traversed to get to those towns
        """
        distances = {towns:float('inf') for towns in self.adjacent}
        distances[start_node] = 0
        previous_nodes = {towns:None for towns in self.adjacent}
        pq=[(0,start_node)]

        while pq:
            current_distance,current_town = heapq.heappop(pq)
            if current_distance > distances[current_town]:
                continue

            for neighbor,time,distance_between in self.adjacent[current_town]:
                # distance = current_distance + float(time[:-1])
                distance = current_distance + float(distance_between[:-2])

                if distance < distances[neighbor]:
                    distances[neighbor] = distance
                    previous_nodes[neighbor] = current_town
                    heapq.heappush(pq,(distance,neighbor))


        return distances,previous_nodes

    def get_shortest_path(self,start_node,end_node):
        """
        - This allows us to get the shortest path after computing the neighbours from dijkstras
        - Provided we know the end node
                - We will perform dijkstras to get the distances and single adjacent towns
                - Specify an empty list called path
                - Traverse the previous_nodes dictionary backwards from the end node until we get to the start node
                        - updating our path list from the front each time
                - We end by providing the distances and the path
        """
        distances,previous_nodes = self.dijkstras(start_node)

        path = []
        current = end_node
        while current is not None:
               path.insert(0,current)
               current = previous_nodes[current]

        return distances[end_node],path

In [48]:
Map = directed_graph()
Map.build()
distances, previous_nodes = Map.dijkstras("Nairobi")
pprint.pprint(distances)
pprint.pprint(previous_nodes)
print("Relevant Knowledge Assembled")

{'Ahero': 393.1,
 'Bahati': 300.5,
 'Bomet': 223.9,
 'Chogoria': 253.40000000000003,
 'Chuka': 237.60000000000002,
 'Dondori': 294.2,
 'Embu': 193.8,
 'Gilgil': 122.5,
 'Githurai': 16.3,
 'Homa Bay': 364.3,
 'Igoji': 263.70000000000005,
 'Juja': 35.0,
 'Kapsoit': 326.8,
 'Karagita': 84.9,
 'Karatina': 138.0,
 'Kendu Bay': 395.5,
 'Kericho': 315.5,
 'Keroka': 280.8,
 'Keumbu': 295.1,
 'Kisii': 307.3,
 'Kisumu': 416.6,
 'Kutus': 174.5,
 'Londiani': 266.5,
 'Mai Mahiu': 53.8,
 'Meru': 297.00000000000006,
 'Molo': 244.39999999999998,
 'Muhoroni': 355.40000000000003,
 'Muiga': 172.8,
 'Nairobi': 0,
 'Naivasha': 92.10000000000001,
 'Nakuru': 164.5,
 'Nanyuki': 373.20000000000005,
 'Narok': 142.9,
 'Ndaragwa': 232.0,
 'Nkubu': 280.40000000000003,
 'Ntulelei': 114.1,
 'Nyahururu': 258.2,
 'Nyeri': 152.0,
 'Othaya': 135.1,
 'Oyugis': 332.6,
 'Rongai': 197.7,
 'Rongo': 333.3,
 'Ruiru': 25.9,
 'Sagana': 112.19999999999999,
 'Sondu': 370.6,
 'Thika': 48.3,
 "Wang'uru": 111.4}
{'Ahero': 'Muhoroni',

### Weather API Integration
Here, we implement a weather API client using the free Open-Meteo API to get real-time weather states (e.g., Sunny, Rainy, Stormy, Foggy) for the key Kenyan towns in our route map.

The Gemini Agent was used to obtain the town coordinates that are present as well as to specify the handling of the weather API.

In [49]:
import requests

class WeatherAPI:
    # Dictionary of known coordinates for backup
    TOWN_COORDINATES = {
        "Nairobi": (-1.2921, 36.8219),
        "Githurai": (-1.2089, 36.9135),
        "Ruiru": (-1.1500, 36.9667),
        "Juja": (-1.1833, 37.0167),
        "Thika": (-1.0333, 37.0667),
        "Othaya": (-0.5431, 36.9419),
        "Sagana": (-0.6667, 37.2000),
        "Wang'uru": (-0.7333, 37.3667),
        "Karatina": (-0.4833, 37.1333),
        "Nyeri": (-0.4167, 36.9500),
        "Kutus": (-0.5167, 37.3333),
        "Embu": (-0.5311, 37.4512),
        "Chuka": (-0.3333, 37.6500),
        "Chogoria": (-0.2333, 37.6333),
        "Igoji": (-0.1500, 37.5833),
        "Nkubu": (-0.0667, 37.6000),
        "Meru": (0.0500, 37.6500),
        "Nanyuki": (0.0167, 37.0667),
        "Muiga": (-0.3500, 36.9000),
        "Ndaragwa": (-0.1167, 36.4333),
        "Nyahururu": (0.0333, 36.3667),
        "Bahati": (-0.2500, 36.2500),
        "Nakuru": (-0.3030, 36.0800),
        "Dondori": (-0.2500, 36.2833),
        "Mai Mahiu": (-0.9833, 36.5833),
        "Karagita": (-0.7333, 36.4000),
        "Naivasha": (-0.7167, 36.4333),
        "Gilgil": (-0.4833, 36.3167),
        "Ntulelei": (-1.0167, 36.0500),
        "Narok": (-1.0781, 35.8601),
        "Bomet": (-0.7833, 35.3500),
        "Keroka": (-0.6333, 34.9500),
        "Keumbu": (-0.6833, 34.8333),
        "Kisii": (-0.6817, 34.7717),
        "Rongo": (-0.7500, 34.6000),
        "Homa Bay": (-0.5273, 34.4571),
        "Kendu Bay": (-0.4333, 34.6500),
        "Ahero": (-0.1744, 34.9203),
        "Oyugis": (-0.5000, 34.7333),
        "Sondu": (-0.3833, 34.9833),
        "Kisumu": (-0.1022, 34.7617),
        "Muhoroni": (-0.1500, 35.2000),
        "Kapsoit": (-0.3167, 35.2167),
        "Kericho": (-0.3677, 35.2827),
        "Londiani": (-0.1667, 35.6000),
        "Molo": (-0.2500, 35.7333),
        "Rongai": (-0.1667, 35.8667)
    }

    # Map complex descriptions to the standard requested elements
    @classmethod
    def normalize_condition(cls, desc):
        desc = desc.lower()
        if any(w in desc for w in ["thunder", "storm", "squall"]):
            return "Stormy"
        elif any(w in desc for w in ["rain", "drizzle", "shower"]):
            return "Rainy"
        elif any(w in desc for w in ["fog", "mist", "haze"]):
            return "Foggy"
        elif any(w in desc for w in ["cloud", "overcast"]):
            return "Cloudy"
        return "Sunny"

    @classmethod
    def get_weather(cls, town):
        town_normalized = town.strip().title()
        if town_normalized not in cls.TOWN_COORDINATES:
            return "Unknown Town"

        # Primary API: WTTR.in (Requests JSON format directly for the specific town in Kenya)
        try:
            url = f"https://wttr.in/{town_normalized},Kenya?format=j1"
            response = requests.get(url, timeout=8)
            if response.status_code == 200:
                data = response.json()
                desc = data["current_condition"][0]["weatherDesc"][0]["value"]
                return cls.normalize_condition(desc)
        except Exception:
            pass

        # Secondary API: Open-Meteo API using coordinate specifications with extended timeout
        try:
            lat, lon = cls.TOWN_COORDINATES[town_normalized]
            url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
            response = requests.get(url, timeout=8)
            if response.status_code == 200:
                data = response.json()
                code = data.get("current_weather", {}).get("weathercode", 0)
                # Map Open-Meteo WMO weather codes to simple elements
                if code in [95, 96, 99]:
                    return "Stormy"
                elif code in [51, 53, 55, 61, 63, 65, 80, 81, 82]:
                    return "Rainy"
                elif code in [45, 48]:
                    return "Foggy"
                elif code in [1, 2, 3]:
                    return "Cloudy"
                return "Sunny"
        except Exception:
            return "API Error"

        return "Service Temporarily Unavailable"

# Quick Demo of the live resilient weather service
print("Nairobi Weather:", WeatherAPI.get_weather("Nairobi"))
print("Kisumu Weather:", WeatherAPI.get_weather("Kisumu"))

Nairobi Weather: Rainy
Kisumu Weather: Rainy


**Step 3: Define the Vocabulary**
*   Predicates (represented in Python as keys and rules): Town, Next Town to go to
*   Constants: Towns such as Nairobi, Naivasha, Gilgil

**Step 4: Encode General Knowledge about the Domain** `[3 marks]`

```
It's crucial for knowledge engineers to explicitly declare the vocabulary here. If you used APIs/web scraping tools to assemble  knowledge for the practical exercise, check step 2.
```

In [51]:
rules = {}

# Gets the shortest path between every different town and creates a knowledge base based on the shortest path it should follow to go to Kisumu.

rules.update({"towns":{}})

for town in previous_nodes:
  if town != "Kisumu":
    distance, path = Map.get_shortest_path(town,"Kisumu")
    rules["towns"].update({town:"Drive to "+path[1]})



print("=========RULES============")
pprint.pprint(rules)

=========RULES============
{'towns': {'Ahero': 'Drive to Kisumu',
           'Bahati': 'Drive to Nakuru',
           'Bomet': 'Drive to Keroka',
           'Chogoria': 'Drive to Igoji',
           'Chuka': 'Drive to Chogoria',
           'Dondori': 'Drive to Nakuru',
           'Embu': 'Drive to Chuka',
           'Gilgil': 'Drive to Nakuru',
           'Githurai': 'Drive to Ruiru',
           'Homa Bay': 'Drive to Kendu Bay',
           'Igoji': 'Drive to Nkubu',
           'Juja': 'Drive to Thika',
           'Kapsoit': 'Drive to Muhoroni',
           'Karagita': 'Drive to Naivasha',
           'Karatina': 'Drive to Nyeri',
           'Kendu Bay': 'Drive to Ahero',
           'Kericho': 'Drive to Kapsoit',
           'Keroka': 'Drive to Keumbu',
           'Keumbu': 'Drive to Kisii',
           'Kisii': 'Drive to Oyugis',
           'Kutus': 'Drive to Embu',
           'Londiani': 'Drive to Kericho',
           'Mai Mahiu': 'Drive to Karagita',
           'Meru': 'Drive to Nanyuki',


**Step 5: Add New Facts to the Knowledge Base** `[4 marks]`

In [53]:
def show_kb():
    print("\n--- Current Knowledge Base ---")
    print("--------------Towns-------------")
    for town, action in rules["towns"].items():
        print(f"{town:10} ➜ {action}")
    print("--------------Weather-------------")
    for weather, action in rules["weather"].items():
        print(f"{weather:19} ➜ {action}")
    print("-------------------------------")

def add_new_town_fact(town, action):
    """Add a new fact directly to the KB (- without user input)."""
    rules["towns"][town.capitalize()] = action
    print(f"Added fact: '{town.capitalize()}' ➜ '{action}'")

# Example of adding facts
add_new_town_fact("Kisumu", "Stop, you have reached your final destination")
add_new_town_fact("Mombasa", "Stay on Mombasa Road and drive to Nairobi")

def add_town_fact():
    """Add new knowledge (facts) to the KB."""
    new_town = input("Enter new Town: ").capitalize()
    new_action = input(f"What should the agent do when the town is {new_town}? ")
    rules["towns"][new_town] = new_action
    print(f" New fact added: '{new_town}' -> '{new_action}'")

add_town_fact()

def add_new_weather_fact(weather, action):
    """Add a new fact directly to the KB (- without user input)."""
    rules["weather"][weather.capitalize()] = action
    print(f"Added fact: '{weather.capitalize()}' ➜ '{action}'")

# Example of adding facts
add_new_weather_fact("Hail", "Stop, Lest you damage your windscreen")
add_new_weather_fact("Cloudy with a chance", "Go outside for meatballs")

def add_weather_fact():
    """Add new knowledge (facts) to the KB."""
    new_weather = input("Enter new Weather: ").capitalize()
    new_action = input(f"What should the agent do when the weather is {new_weather}? ")
    rules["weather"][new_weather] = new_action
    print(f" New fact added: '{new_weather}' -> '{new_action}'")

add_town_fact()

# View updated KB
show_kb()


Added fact: 'Kisumu' ➜ 'Stop, you have reached your final destination'
Added fact: 'Mombasa' ➜ 'Stay on Mombasa Road and drive to Nairobi'
Enter new Town: London
What should the agent do when the town is London? Go to Heathrow and deposit your cargo to be transported to JKIA
 New fact added: 'London' -> 'Go to Heathrow and deposit your cargo to be transported to JKIA'
Added fact: 'Hail' ➜ 'Stop, Lest you damage your windscreen'
Added fact: 'Cloudy with a chance' ➜ 'Go outside for meatballs'
Enter new Town: Muddy
What should the agent do when the town is Muddy? Take a muddy shower
 New fact added: 'Muddy' -> 'Take a muddy shower'

--- Current Knowledge Base ---
--------------Towns-------------
Nairobi    ➜ Drive to Mai Mahiu
Githurai   ➜ Drive to Ruiru
Ruiru      ➜ Drive to Juja
Juja       ➜ Drive to Thika
Thika      ➜ Drive to Othaya
Othaya     ➜ Drive to Nyeri
Sagana     ➜ Drive to Karatina
Wang'uru   ➜ Drive to Kutus
Karatina   ➜ Drive to Nyeri
Nyeri      ➜ Drive to Muiga
Kutus      

**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers** `[3 marks]`



In [54]:
# Step 6: The current observation from sensors/camera
current_town = input("Enter the current town: ").capitalize()
current_weather = WeatherAPI.get_weather(current_town)


# Step 7: The agent queries the knowledge base (rules dictionary)
def town_advice(town):
    action = rules["towns"].get(town, "Unknown town — proceed to nearest Self Driving Truck Service Station.")
    return action

def weather_advice(weather):
    action = rules["weather"].get(weather, "Unknown weather pattern — proceed but with heightened caution and reflexes.")
    return action

# Output the result
print("\n=== Self Driving Truck Decision System ===")
print(f"Observed Light: {current_town}")
print(f"Current weather is: ", current_weather, " ")
print("Agent Decision:", town_advice(current_town),", ", weather_advice(current_weather))

Enter the current town: Nakuru

=== Self Driving Truck Decision System ===
Observed Light: Nakuru
Current weather is:  Rainy  
Agent Decision: Drive to Rongai ,  Drive at low speed (High hydroplaning risk)


**Step 8: Querying the knowledge base**

In [55]:
#ask questions to get actions based on known facts (in this case; towns)
print("Agent Decision for 'Igoji':", town_advice("Igoji"))
print("Agent Decision for 'Nairobi':", town_advice("Nairobi"))
print("Agent Decision for 'Sao Paulo':", town_advice("Sao Paulo"))

#ask questions to get actions based on known facts (in this case; weather)
print("Agent Decision for 'Sunny':", weather_advice("Sunny"))
print("Agent Decision for 'Rainy':", weather_advice("Rainy"))
print("Agent Decision for 'Cloudy with a chance':", weather_advice("Cloudy with a chance"))

Agent Decision for 'Igoji': Drive to Nkubu
Agent Decision for 'Nairobi': Drive to Mai Mahiu
Agent Decision for 'Sao Paulo': Unknown town — proceed to nearest Self Driving Truck Service Station.
Agent Decision for 'Sunny': Unknown weather pattern — proceed but with heightened caution and reflexes.
Agent Decision for 'Rainy': Drive at low speed (High hydroplaning risk)
Agent Decision for 'Cloudy with a chance': Go outside for meatballs


**Step 9: Debugging the Knowledge Base** `[2 marks]`

In [56]:
# You can test new or faulty signals to see how the agent reacts.
test_cases = ["Nairobi", "Embu", "Ahero", "Gilgil", "Narok", "Isiolo", "Abidjan", "Mogadishu"]
print("\n--- Debugging the Knowledge Base ---")
for town in test_cases:
    print(f"Town: {town:12} -> Action: {town_advice(town)}")

# You can test new or faulty signals to see how the agent reacts.
weather_cases = ["Sunny/Clear", "Rainy", "Cloudy", "Partly cloudy", "Hail", "Thunderstorm", "Thunderstruck", "Muddy"]
print("\n--- Debugging the Knowledge Base ---")
for weather in weather_cases:
    print(f"Weather: {weather:18} -> Action: {weather_advice(weather)}")


--- Debugging the Knowledge Base ---
Town: Nairobi      -> Action: Drive to Mai Mahiu
Town: Embu         -> Action: Drive to Chuka
Town: Ahero        -> Action: Drive to Kisumu
Town: Gilgil       -> Action: Drive to Nakuru
Town: Narok        -> Action: Drive to Bomet
Town: Isiolo       -> Action: Unknown town — proceed to nearest Self Driving Truck Service Station.
Town: Abidjan      -> Action: Unknown town — proceed to nearest Self Driving Truck Service Station.
Town: Mogadishu    -> Action: Unknown town — proceed to nearest Self Driving Truck Service Station.

--- Debugging the Knowledge Base ---
Weather: Sunny/Clear        -> Action: Drive at regular speed (Normal limits)
Weather: Rainy              -> Action: Drive at low speed (High hydroplaning risk)
Weather: Cloudy             -> Action: Drive at regular speed, maintain alert state
Weather: Partly cloudy      -> Action: Unknown weather pattern — proceed but with heightened caution and reflexes.
Weather: Hail               -> Ac

You can then add or edit an action to whatever object based on the test cases. e.g light none then continue driving

**Step 10: Add final reflection here (challenges or recommended improvements)** `[3 marks]`

Initial challenges were on how to develop the knowledge base but settled on using the shortest distance rather than using the first path seen.

Recommended Improvements would include using as many towns as possible, rather than just limiting to the towns that have been showcased.


![Image that is there](https://drive.google.com/uc?export=view&id=16xnNEiLqEutwoewcMwVx3OZgfvC-dFth)
